In [1]:
import duckdb
import os
from pathlib import Path

# 1. Identify the current directory (Handles both .py scripts and Jupyter)
try:
    # Use __file__ for standalone scripts
    current_location = Path(__file__).resolve().parent
except NameError:
    # Use Current Working Directory for Jupyter/Interactive sessions
    current_location = Path.cwd().resolve()

# 2. Navigate to the actual Project Root
# If current_location is 'Finalized_Scripts', move to the parent directory
if current_location.name in ["Finalized_Scripts", "Test_Scripts", "scripts"]:
    PROJECT_ROOT = current_location.parent
else:
    PROJECT_ROOT = current_location

# 3. Define the absolute path to the data
# This ensures the path is /floodnet_work/Data_Files/ instead of /Finalized_Scripts/Data_Files/
data_dir = PROJECT_ROOT / "Data_Files"
file_name = "delineated_storms.parquet"
file_path = data_dir / file_name

# 4. Safety Check and Data Loading
if not file_path.exists():
    # This will print the exact path being searched to help with troubleshooting
    raise FileNotFoundError(f"Target file not found at: {file_path}")

# Resolve the directory path
data_dir = PROJECT_ROOT / "Data_Files"

# Define specific filenames
data_file_name = "floodnet_full_dataset_merged_with_weather.parquet"
output_filename = "delineated_storms.parquet"

# Resolve full paths as strings for compatibility with data readers
data_path = str(data_dir / data_file_name)
output_path = str(data_dir / output_filename)

con = duckdb.connect()
con.execute("SET threads TO 2;")
con.execute("SET memory_limit = '180GB';")
con.execute("SET temp_directory = './tmp_duckdb';")

MIT_SECONDS = 21600
LEAD_TIME_HOURS = 2
LAG_TIME_HOURS = 6

MIN_WET_THRESHOLD   = 0.01
MIN_TOTAL_PRECIP    = 0.25
MIN_PEAK_INTENSITY  = 0.10
MIN_DEPTH_RISE      = 1.0

# ✅ Weather timestamp column in your merged dataset:
WEATHER_TIME_COL = '"DATE"'  # quoted because DATE can be finicky

print("🚀 Starting Storm Delineation (Merged Windows + Phase Labels)...")
print(f"   Buffers: -{LEAD_TIME_HOURS}h lead | +{LAG_TIME_HOURS}h lag")
print(f"   MIT gap: {MIT_SECONDS/3600:.1f} hours")
print(f"   Weather time column: {WEATHER_TIME_COL}")

# ✅ minutes_since_weather_update expression
if WEATHER_TIME_COL is None:
    weather_age_expr = "NULL::DOUBLE AS minutes_since_weather_update"
else:
    weather_age_expr = f"""
        CASE
          WHEN d.{WEATHER_TIME_COL} IS NULL THEN NULL::DOUBLE
          ELSE (epoch(d.time) - epoch(d.{WEATHER_TIME_COL})) / 60.0
        END AS minutes_since_weather_update
    """


# ================================
# Build the SQL expression for minutes_since_weather_update if enabled
# ================================
if WEATHER_TIME_COL is None:
    weather_age_expr = "NULL::DOUBLE AS minutes_since_weather_update"
else:
    weather_age_expr = f"""
        CASE
          WHEN {WEATHER_TIME_COL} IS NULL THEN NULL::DOUBLE
          ELSE (epoch(d.time) - epoch(d.{WEATHER_TIME_COL})) / 60.0
        END AS minutes_since_weather_update
    """

query = f"""
COPY (
    WITH wet_records AS (
        -- Step 1: identify wet minutes using incremental precip
        SELECT
            deployment_id,
            time,
            LAG(time) OVER (PARTITION BY deployment_id ORDER BY time) AS prev_wet_time
        FROM '{data_path}'
        WHERE "precip_incremental [inch]" >= {MIN_WET_THRESHOLD}
    ),

    wet_storm_ids AS (
        -- Step 2: assign preliminary storm ids based on MIT gap between wet minutes
        SELECT
            deployment_id,
            time,
            SUM(
                CASE 
                    WHEN prev_wet_time IS NULL THEN 1
                    WHEN (epoch(time) - epoch(prev_wet_time)) > {MIT_SECONDS} THEN 1
                    ELSE 0
                END
            ) OVER (
                PARTITION BY deployment_id
                ORDER BY time
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            ) AS storm_id_raw
        FROM wet_records
    ),

    raw_windows AS (
        -- Step 3: compute rain start/end (unbuffered) and storm start/end (buffered)
        SELECT
            deployment_id,
            storm_id_raw,
            MIN(time) AS rain_start,
            MAX(time) AS rain_end,
            MIN(time) - INTERVAL '{LEAD_TIME_HOURS} hours' AS storm_start,
            MAX(time) + INTERVAL '{LAG_TIME_HOURS} hours' AS storm_end
        FROM wet_storm_ids
        GROUP BY deployment_id, storm_id_raw
    ),

    ordered AS (
        -- Step 4: order storm windows and look back to detect overlap
        SELECT
            *,
            LAG(storm_end) OVER (PARTITION BY deployment_id ORDER BY storm_start) AS prev_end
        FROM raw_windows
    ),

    flags AS (
        -- Step 5: mark a new merged group if current start is after previous end (no overlap)
        SELECT
            *,
            CASE
                WHEN prev_end IS NULL THEN 1
                WHEN storm_start > prev_end THEN 1
                ELSE 0
            END AS new_group_flag
        FROM ordered
    ),

    merged_ids AS (
        -- Step 6: cumulative sum to create merged storm groups (prevents overlap duplication)
        SELECT
            *,
            SUM(new_group_flag) OVER (PARTITION BY deployment_id ORDER BY storm_start) AS storm_id
        FROM flags
    ),

    final_windows AS (
        -- Step 7: merge windows per deployment_id + merged storm_id
        SELECT
            deployment_id,
            storm_id,
            MIN(storm_start) AS storm_start,
            MAX(storm_end) AS storm_end,
            MIN(rain_start) AS rain_start,
            MAX(rain_end) AS rain_end
        FROM merged_ids
        GROUP BY deployment_id, storm_id
    ),

    all_records_in_storms AS (
        -- Step 8: join all depth rows that fall inside the merged buffered windows
        SELECT
            d.*,
            sw.storm_id,
            d.deployment_id || '_' || sw.storm_id AS global_storm_id,
            sw.storm_start,
            sw.storm_end,
            sw.rain_start,
            sw.rain_end,
            CASE
                WHEN d.time < sw.rain_start THEN 'lead'
                WHEN d.time <= sw.rain_end THEN 'storm'
                ELSE 'lag'
            END AS phase,
            (epoch(d.time) - epoch(sw.rain_start)) / 3600.0 AS hours_since_rain_start,
            (epoch(d.time) - epoch(sw.storm_start)) / 3600.0 AS hours_since_storm_start,
            {weather_age_expr}
        FROM '{data_path}' d
        INNER JOIN final_windows sw
            ON d.deployment_id = sw.deployment_id
            AND d.time BETWEEN sw.storm_start AND sw.storm_end
    ),

    storm_metrics AS (
        -- Step 9: compute storm-level metrics on the buffered (merged) window
        SELECT
            global_storm_id,
            deployment_id,
            storm_id,
            SUM("precip_incremental [inch]") AS total_precip_in,
            MAX("precip_max_intensity [inch/hour]") AS peak_intensity_inh,
            MAX(depth_inches) - MIN(depth_inches) AS net_depth_rise_in,
            COUNT(*) AS storm_record_count,
            (epoch(MAX(storm_end)) - epoch(MIN(storm_start))) / 3600.0 AS storm_duration_hr
        FROM all_records_in_storms
        GROUP BY global_storm_id, deployment_id, storm_id
    ),

    significant_storms AS (
        -- Step 10: filter significant storms (same logic as your original)
        SELECT *
        FROM storm_metrics
        WHERE total_precip_in >= {MIN_TOTAL_PRECIP}
           OR peak_intensity_inh >= {MIN_PEAK_INTENSITY}
           OR (net_depth_rise_in >= {MIN_DEPTH_RISE} AND COALESCE(total_precip_in, 0) >= 0.05)
    )

    -- Step 11: final output rows for significant storms
    SELECT
        a.*,
        s.total_precip_in,
        s.peak_intensity_inh,
        s.net_depth_rise_in,
        s.storm_record_count,
        s.storm_duration_hr
    FROM all_records_in_storms a
    INNER JOIN significant_storms s
        ON a.global_storm_id = s.global_storm_id
    ORDER BY a.deployment_id, a.time

) TO '{output_path}' (FORMAT PARQUET, COMPRESSION 'ZSTD');
"""

con.execute(query)
print(f"✅ Delineation complete → {output_path}")

# ================================
# Post-check: duplicates test
# ================================
dupes = con.execute(f"""
    SELECT deployment_id, time, COUNT(*) AS n
    FROM '{output_path}'
    GROUP BY deployment_id, time
    HAVING COUNT(*) > 1
    ORDER BY n DESC
    LIMIT 20;
""").df()

print("\n🔎 Duplicate row check (should be empty):")
print(dupes.to_string(index=False) if len(dupes) else "No duplicates found ✅")

# ================================
# Summary
# ================================
summary = con.execute(f"""
    SELECT
        COUNT(DISTINCT deployment_id) AS sensors,
        COUNT(DISTINCT global_storm_id) AS total_storms,
        ROUND(AVG(total_precip_in), 3) AS avg_precip_in,
        ROUND(AVG(storm_duration_hr), 2) AS avg_duration_hr,
        ROUND(AVG(storm_record_count), 1) AS avg_records_per_storm
    FROM '{output_path}';
""").df()

print("\n📊 Storm Summary:")
print(summary.to_string(index=False))

🚀 Starting Storm Delineation (Merged Windows + Phase Labels)...
   Buffers: -2h lead | +6h lag
   MIT gap: 6.0 hours
   Weather time column: "DATE"


RuntimeError: Query interrupted

In [2]:
check = con.execute(f"""
    SELECT
        MIN(minutes_since_weather_update) AS min_age,
        APPROX_QUANTILE(minutes_since_weather_update, 0.50) AS p50_age,
        APPROX_QUANTILE(minutes_since_weather_update, 0.95) AS p95_age,
        MAX(minutes_since_weather_update) AS max_age
    FROM '{output_path}';
""").df()

print(check.to_string(index=False))

 min_age  p50_age  p95_age  max_age
     0.0 2.506208 4.745733 4.999989


In [3]:
neg = con.execute(f"""
    SELECT COUNT(*) AS n_negative
    FROM '{output_path}'
    WHERE minutes_since_weather_update < -1e-6;
""").df()

print(neg.to_string(index=False))

 n_negative
          0
